# Lab: Coding a Simple Linear Regression Workflow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/okuchap/GB656_2026_public/blob/main/problem-sets/lab-lectures/02-01-lab.ipynb)

This lab prepares you for Problem Set 1 by making the scientific Python workflow predictable. We practice with **mileage** as the feature and listing price as the outcome. Explanations focus on coding techniques used for data analysis, modeling, and prediction rather than on general Python grammar.

## Lab goals

By the end of the lab, you should be able to:

- load a course CSV from a local copy or the public GitHub repository;
- inspect, select, rename, and derive columns with `pandas`;
- make a scatterplot with `matplotlib`;
- prepare `y` and `X` and include an intercept;
- fit OLS with `statsmodels`;
- extract coefficients, standard errors, p-values, confidence intervals, and $R^2$;
- create a correctly shaped table of new observations and predict from it; and
- diagnose common coding mistakes.

## How to use this notebook

Open the lab using the course Google Colab link and work from top to bottom. Before running a code cell, read the explanation immediately above it and try to say what each line will create or display. After running it, compare the output with the explanation or **Check** note.

You do not submit this lab. If you want your changes to persist after you close Colab, select **File > Save a copy in Drive**; otherwise, saving a copy is optional.

If an exercise invites you to change a value, return it to a valid value before continuing so later cells still run. A name created in one cell, such as `cars`, remains available to later cells only after the earlier cell has run.

If you run into an error, first check Section 10, **Common mistakes and quick diagnoses**.

## 0. Setup

Google Colab already includes these packages, so no installation command is needed in a standard Colab runtime.

- `numpy` (`np`) supplies numerical utilities, `pandas` (`pd`) supplies tables, `matplotlib.pyplot` (`plt`) supplies plots, and `statsmodels.api` (`sm`) supplies the regression model.
- `plt.style.use(...)` applies a consistent visual style to plots; it does not change the data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from pathlib import Path
from urllib.parse import quote

plt.style.use("seaborn-v0_8-whitegrid")

## 1. Load a CSV safely

`pd.read_csv()` loads the course CSV into a `pandas` **DataFrame**, a two-dimensional table with labeled rows and columns. It accepts either a local file path or a web URL.

The instructor-provided `course_data_source()` helper first searches the current directory and its parent directories for `data/car_price_prediction.csv`. If it cannot find a local copy—as in a fresh Colab runtime—it returns the raw-data URL from the public course repository on GitHub. You do not need to upload the CSV or mount Google Drive.

You do not need to understand the helper's internal path-searching code for this lab. Focus on how its returned path or URL becomes the input to `pd.read_csv()`. `cars_raw.shape` reports the number of rows and columns, while `cars_raw.head()` previews the first five observations. These are useful first checks that the expected dataset was loaded.

In [ ]:
PUBLIC_REPOSITORY = "okuchap/GB656_2026_public"
PUBLIC_REVISION = "main"


def course_data_source(file_name):
    """Return a local course-data path when available, otherwise its public URL."""
    for root in (Path.cwd(), *Path.cwd().parents):
        local_path = root / "data" / file_name
        if local_path.is_file():
            return local_path

    encoded_name = quote(file_name)
    return (
        "https://raw.githubusercontent.com/"
        f"{PUBLIC_REPOSITORY}/{PUBLIC_REVISION}/data/{encoded_name}"
    )


data_source = course_data_source("car_price_prediction.csv")
cars_raw = pd.read_csv(data_source)
source_location = (
    "local course repository" if isinstance(data_source, Path) else "public GitHub repository"
)
print(
    f"Loaded {cars_raw.shape[0]:,} rows and {cars_raw.shape[1]} columns "
    f"from the {source_location}."
)
cars_raw.head()

### What to check immediately

- `head()` previews observations.
- `shape` reports `(number of rows, number of columns)`.
- `columns.tolist()` reports exact column names, which must match later selections.
- `info()` reports column data types and non-null counts.

Together, these checks help catch an incorrect dataset, unexpected schema, nonnumeric modeling variables, or missing values before analysis begins.

In [ ]:
print("Shape:", cars_raw.shape)
print("Columns:", cars_raw.columns.tolist())
cars_raw.info()

### Exercise 1 — inspect before modeling

Before running the next cell, predict what it will show. `cars_raw[["Mileage", "Price"]]` passes a Python list of two column names inside the outer selection brackets, so the result remains a two-column DataFrame. `.describe()` computes summary statistics for its numeric columns. `.round(2)` returns a display-friendly copy rounded to two decimal places; it does not alter `cars_raw`.

This is also a short example of **method chaining**: the value returned by `.describe()` becomes the object on which `.round(2)` runs. Then check:

1. Are `Mileage` and `Price` numeric?
2. Are there any missing values in those columns?
3. Does `Mileage` appear to be measured in miles rather than thousands of miles?

In [ ]:
cars_raw[["Mileage", "Price"]].describe().round(2)

**Check:** You should see 1,000 nonmissing values in each column. Mileage extends to roughly 200,000, so its original unit is miles.

## 2. Select and rename columns

Double brackets select a table of columns. Single brackets around one name, such as `cars_raw["Mileage"]`, return a one-dimensional Series; double brackets around a list, such as `cars_raw[["Mileage", "Price"]]`, return a two-dimensional DataFrame.

The cleaning operations are chained so that each method operates on the DataFrame returned by the previous step:

1. `.dropna()` removes rows containing a missing value in either selected column. It returns a cleaned DataFrame and does not modify `cars_raw`.
2. `.rename(columns={...})` maps the original column names to shorter analysis names.
3. `.copy()` creates an independent working table. This makes it safe to add columns to `cars` without accidentally changing a view of `cars_raw`.

The final `cars.head()` confirms the result of the entire chain.

In [ ]:
cars = (
    cars_raw[["Mileage", "Price"]]
    .dropna()
    .rename(columns={"Mileage": "mileage", "Price": "price"})
    .copy()
)

cars.head()

### Exercise 2 — column selection syntax

The next cell selects a different pair of columns for practice. Notice that `"Engine Size"` must match the original spelling and space exactly. The result is saved separately, so it does not change `cars_raw`.

In [ ]:
practice_columns = cars_raw[["Engine Size", "Price"]].copy()  # TODO: run and inspect
practice_columns.head()

**Check:** The result should be a two-column DataFrame. A `KeyError` here usually means a column name was misspelled.

## 3. Create a derived feature

Raw mileage gives a slope in dollars per one mile, an inconveniently small unit. Dividing by 1,000 creates a feature measured in thousands of miles.

`cars["mileage"]` selects one Series. Dividing that Series by `1000` is **vectorized**: `pandas` divides every value without requiring a loop. `cars["mileage_1000"] = ...` assigns the resulting Series to a new column. The number of rows does not change. The second line selects three columns and previews them so that we can verify the calculation.

In [ ]:
cars["mileage_1000"] = cars["mileage"] / 1000
cars[["mileage", "mileage_1000", "price"]].head()

### Exercise 3 — make another unit conversion

Create mileage measured in tens of thousands of miles. The line is complete so the notebook remains runnable; explain to a partner why dividing by 10,000 changes the unit but not the underlying cars.

Compare the new column with `mileage_1000`. The same observations are represented in different units, so regression coefficients using these two features would have different numerical scales.

In [ ]:
cars["mileage_10_000"] = cars["mileage"] / 10_000  # TODO: run and explain the unit
cars[["mileage", "mileage_1000", "mileage_10_000"]].head()

### Check the conversion with an assertion

`np.allclose(a, b)` checks whether all corresponding numeric values are equal within a small floating-point tolerance. Here it verifies the expected relationship between the two rescaled mileage features. Tolerance-based comparison is safer than exact equality for scientific calculations involving decimal values.

In [ ]:
assert np.allclose(cars["mileage_1000"], 10 * cars["mileage_10_000"])
print("Unit-conversion check passed.")

## 4. Make and read a scatterplot

For simple linear regression, put the feature on the horizontal axis and the outcome on the vertical axis.

The plotting code uses Matplotlib's object-oriented interface:

- `plt.subplots(figsize=(7, 5))` creates the Figure and plotting area (Axes) and sets the figure size.
- `ax.scatter(x, y, ...)` draws one point for each paired value. The first argument supplies horizontal positions and the second supplies vertical positions. `alpha=0.5` makes points 50% opaque so overlapping observations are easier to see.
- `ax.set_xlabel()`, `ax.set_ylabel()`, and `ax.set_title()` attach the labels needed to interpret variables and units.
- `plt.show()` renders the completed figure.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(cars["mileage_1000"], cars["price"], alpha=0.5)
ax.set_xlabel("Mileage (thousands of miles)")
ax.set_ylabel("Listing price ($)")
ax.set_title("Used-car listing price vs. mileage")

plt.show()

### Exercise 4 — read the plot

Write one sentence for each question in your notes:

1. Is the overall direction positive, negative, or flat?
2. Does a straight-line summary seem useful?
3. Does mileage explain every price difference?

**Hint:** Look at both the left-to-right direction and the vertical spread at similar mileage values.

## 5. Prepare `y` and `X`

`statsmodels` keeps the outcome and feature table separate:

- `y = cars["price"]` uses single brackets and therefore creates the one-dimensional outcome Series expected by `statsmodels`.
- `X = cars[["mileage_1000"]]` uses double brackets and therefore creates a two-dimensional feature DataFrame—even though it contains only one feature. Many modeling tools require this table shape.
- `sm.add_constant(X)` returns a new DataFrame with a column named `const`, filled with 1s. Multiplying its coefficient by 1 adds the intercept to every fitted value.

The second assignment reuses the name `X`: after `X = sm.add_constant(X)`, `X` refers to the new two-column DataFrame.

In [ ]:
y = cars["price"]
X = cars[["mileage_1000"]]

print("Before adding the intercept:")
display(X.head())

X = sm.add_constant(X)

print("After adding the intercept:")
display(X.head())

### Exercise 5 — check dimensions

A model requires one feature row for every outcome value. These checks verify that `y` and `X` have the same number of rows and that `X` contains the expected columns in the expected order: `const` followed by `mileage_1000`. A failed check signals that the model inputs were constructed incorrectly.

In [ ]:
assert len(y) == len(X)
assert X.columns.tolist() == ["const", "mileage_1000"]
print("Model-input checks passed.")

## 6. Fit OLS and print the regression summary

Ordinary least squares chooses the intercept and slope that minimize the sum of squared residuals.

`sm.OLS(y, X)` constructs an OLS model specification from the outcome and design matrix but has not estimated coefficients yet. `.fit()` performs the estimation and returns a fitted-results object. The assignment stores that object as `mileage_model` so we can reuse its many results without refitting.

`mileage_model.summary()` produces the full statistical report. For this course, focus on the `const` and `mileage_1000` rows plus $R^2$; later cells show how to retrieve those values directly.

In [ ]:
mileage_model = sm.OLS(y, X).fit()
print(mileage_model.summary())

## 7. Extract the important results

A fitted `statsmodels` results object stores outputs in named attributes. `params`, `bse`, and `pvalues` are Series indexed by coefficient name, so `["mileage_1000"]` retrieves the value for the slope rather than depending on its numerical position. `rsquared` is a single number.

`conf_int()` returns a two-column DataFrame of confidence-interval endpoints. `.loc["mileage_1000"]` selects the slope row by its coefficient label, while `.iloc[0]` and `.iloc[1]` retrieve its lower and upper endpoints. Label-based selection makes the extraction robust to changes in coefficient order.

In [ ]:
intercept = mileage_model.params["const"]
slope = mileage_model.params["mileage_1000"]
slope_se = mileage_model.bse["mileage_1000"]
slope_p_value = mileage_model.pvalues["mileage_1000"]
slope_ci = mileage_model.conf_int().loc["mileage_1000"]
r_squared = mileage_model.rsquared

print(f"Intercept:              ${intercept:,.2f}")
print(f"Slope per 1,000 miles:  ${slope:,.2f}")
print(f"Slope standard error:   ${slope_se:,.2f}")
print(f"Slope p-value:           {slope_p_value:.4g}")
print(f"Slope 95% CI:           [${slope_ci.iloc[0]:,.2f}, ${slope_ci.iloc[1]:,.2f}]")
print(f"R-squared:               {r_squared:.3f}")

### Assemble selected outputs into a table

The next cell presents the same model results in a smaller DataFrame. Each model-output Series becomes a column. Because the Series share the coefficient labels `const` and `mileage_1000`, pandas aligns their rows by those labels.

`conf_int[0]` and `conf_int[1]` supply the lower and upper endpoints. `.rename(index={"const": "intercept"})` gives the intercept row a more descriptive label. Finally, `.round(4)` creates a display-friendly table without changing the more precise model results.

In [ ]:
conf_int = mileage_model.conf_int()

coef_table = pd.DataFrame(
    {
        "estimate": mileage_model.params,
        "std_error": mileage_model.bse,
        "p_value": mileage_model.pvalues,
        "ci_lower": conf_int[0],
        "ci_upper": conf_int[1],
    }
).rename(index={"const": "intercept"})

coef_table.round(4)

### Interpretation templates

- **Intercept:** predicted outcome when the feature equals 0. First check whether 0 is meaningful and reasonably close to the observed feature range.
- **Slope:** each one-unit increase in the feature is associated with an estimated slope-sized change in the predicted outcome, on average.
- **Standard error:** how much the estimate would tend to vary across repeated samples under the model assumptions.
- **P-value:** for the slope row, this tests $H_0: \beta_1=0$. The p-value is the probability of obtaining a test statistic at least as extreme as the observed one across repeated samples *if the null hypothesis $H_0$ is true*. Thus, a small p-value is evidence against $H_0$
  - Be careful about the definition: for example, $p$-value is *not* the probability that the null hypothesis is true.
- **95% confidence interval:** across repeated samples, about 95% of intervals constructed this way would contain the true population coefficient $\beta_1$ under the model assumptions.

### Exercise 6 — interpret the mileage slope

Fill in this sentence in your notes using the printed result:

> Each additional ______ miles is associated with about a $______ change in predicted listing price, on average.

**Check:** The unit is 1,000 miles. Because the fitted slope is negative, the predicted price change is a decrease. The data are observational, so use association language rather than a causal claim.

## 8. Make predictions for new observations

Prediction data must use the same feature name and the same units as the fitted model. The next cell follows the same data-shape rules used for fitting:

- `pd.DataFrame({"mileage_1000": [25, 75, 125]})` builds a three-row prediction table whose feature name matches the training data.
- Multiplying the `mileage_1000` Series by `1000` converts all three values back to miles and assigns the result to a new display column. The model still uses `mileage_1000`.
- `sm.add_constant(..., has_constant="add")` creates the same intercept column used during fitting.
- `mileage_model.predict(new_X)` returns one predicted outcome for every row of `new_X`. Assigning that Series creates the `predicted_price` column.
- The last line selects the columns in a reader-friendly order and rounds their displayed values.

In [ ]:
new_cars = pd.DataFrame({"mileage_1000": [25, 75, 125]})
new_cars["mileage"] = new_cars["mileage_1000"] * 1000

new_X = sm.add_constant(
    new_cars[["mileage_1000"]], has_constant="add"
)
new_cars["predicted_price"] = mileage_model.predict(new_X)

new_cars[["mileage", "mileage_1000", "predicted_price"]].round(2)

**Why force the constant?** `sm.add_constant()` normally examines the supplied values and decides whether a constant column is already present. With a small prediction table—especially a one-row table—a feature can look constant by accident. `has_constant="add"` tells the function to add `const` unconditionally, giving `new_X` the same columns as the fitted `X`.

### Exercise 7 — make one new prediction

The cell uses 60,000 miles as a runnable example. Change `MY_MILEAGE` to another nonnegative whole number and rerun it. The value is converted to the model's thousands-of-miles unit and placed in a one-row DataFrame. The feature name and intercept must match the data used to fit the model.

In [ ]:
MY_MILEAGE = 60000  # TODO: try another nonnegative whole-number mileage

my_listing = pd.DataFrame({"mileage_1000": [MY_MILEAGE / 1000]})
my_listing_X = sm.add_constant(
    my_listing[["mileage_1000"]], has_constant="add"
)
my_listing["predicted_price"] = mileage_model.predict(my_listing_X)
my_listing

### Exercise 8 — verify prediction by hand

For a simple line, prediction also equals `intercept + slope * feature`. This check catches unit or column mistakes.

`.loc[0, "mileage_1000"]` retrieves the feature value from the prediction row. `np.isclose(a, b)` verifies that the manual calculation and `statsmodels` prediction agree within a small floating-point tolerance.

In [ ]:
manual_prediction = intercept + slope * my_listing.loc[0, "mileage_1000"]
model_prediction = my_listing.loc[0, "predicted_price"]

assert np.isclose(manual_prediction, model_prediction)
print(f"Both methods give ${model_prediction:,.2f}.")

## 9. Plot the fitted line and predictions

To draw a line, Matplotlib needs ordered horizontal positions and their fitted vertical values. `Series.min()` and `Series.max()` return the endpoints of the observed mileage range. `np.linspace(start, stop, 100)` creates 100 evenly spaced values from the start through the stop. The expression `intercept + slope * x_grid` is vectorized, producing one fitted value for every grid value.

The cell then layers several marks on the same `ax` object:

- `ax.scatter(...)` draws the observed listings.
- `ax.plot(...)` connects the ordered grid points to form the fitted line.
- A second `ax.scatter(...)` adds the predictions using a distinct color and marker.
- Each layer's `label=` supplies text for `ax.legend()`, which identifies the observed data, fitted line, and point predictions.

In [ ]:
x_grid = np.linspace(cars["mileage_1000"].min(), cars["mileage_1000"].max(), 100)
y_grid = intercept + slope * x_grid

fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(cars["mileage_1000"], cars["price"], alpha=0.25, label="Observed listings")
ax.plot(x_grid, y_grid, color="black", linewidth=2.5, label="Fitted line")
ax.scatter(
    new_cars["mileage_1000"],
    new_cars["predicted_price"],
    color="red",
    marker="x",
    s=120,
    linewidths=3,
    label="Point predictions",
)
ax.set_xlabel("Mileage (thousands of miles)")
ax.set_ylabel("Listing price ($)")
ax.set_title("Mileage model: fitted line and point predictions")
ax.legend()

plt.show()

## 10. Common mistakes and quick diagnoses

| Symptom | Likely cause | Fix |
|---|---|---|
| Data-loading or HTTP error | The local CSV is unavailable and the public GitHub file could not be reached | Rerun the instructor-provided load cell, confirm that the repository and filename were not edited, and check the runtime's internet connection |
| `KeyError` | Column spelling/capitalization is wrong | Print `cars_raw.columns.tolist()` and copy the exact name |
| Model has no `const` row | Intercept was not added | Run `X = sm.add_constant(X)` before fitting |
| Prediction reports a shape error | New data do not match fitted columns | Add the constant and use the same feature name/order |
| Slope interpretation has the wrong size | Original and transformed units were mixed | State the unit (`mileage_1000`) before interpreting |
| Claim says the feature “causes” price | Association was mistaken for causation | Use “is associated with” or “predicts” |
| Notebook works only out of order | Hidden state from earlier experiments | Restart the runtime and run all cells from top to bottom |

## 11. Final transfer exercise

Outline the steps you would follow for a new numerical feature:

1. Identify the original outcome and feature columns.
2. Select, rename, and clean them.
3. Create any derived feature and state its unit.
4. Make a scatterplot.
5. Build `y`, build `X`, and add the intercept.
6. Fit OLS and inspect the regression table.
7. Interpret the estimated coefficients and uncertainty statistics.
8. Build new observations with matching columns and units.
9. Make point predictions.
10. Restart the runtime and run all cells from top to bottom.

If you can explain why each step is needed, you are ready for the problem set.

## 12. Tips for Problem Set 1

You've now completed essentially the same modeling workflow you will use in Problem Set 1. The main difference is the feature:

| In this lab | In Problem Set 1 |
|---|---|
| `Mileage` → `mileage_1000` | `Year` → `car_age` |
| `price` is the outcome | Same |
| build `y` and a one-column `X` | Same |
| add an intercept with `sm.add_constant()` | Same |
| fit OLS with `sm.OLS(y, X).fit()` | Same |
| interpret the slope, uncertainty, and $R^2$ | Same |
| build a matching prediction DataFrame | Same |

The main new idea you will transfer is **changing the feature while keeping the modeling workflow the same**.